# ollama 使用介绍

In [1]:
# 安装依赖包
# !pip install ollama langchain langchain_community requests
# !which python

/Users/chan/projects/Itcast_qa_system/.venv/bin/python


## python中使用ollama

In [1]:
import ollama

# 聊天式
response = ollama.chat(model="qwen2.5:7b", messages=[{"role": "user", "content": "你好"}])
print(response)
print(response["message"])

model='qwen2.5:7b' created_at='2026-06-15T09:22:16.609319Z' done=True done_reason='stop' total_duration=1235583292 load_duration=120095208 prompt_eval_count=30 prompt_eval_duration=87693000 eval_count=16 eval_duration=1012012000 message=Message(role='assistant', content='你好！很高兴能为你服务。有什么问题或者需要帮助的吗？', thinking=None, images=None, tool_name=None, tool_calls=None)
role='assistant' content='你好！很高兴能为你服务。有什么问题或者需要帮助的吗？' thinking=None images=None tool_name=None tool_calls=None


In [2]:
## 远程调用的方式
from ollama import Client

client = Client(host="http://127.0.0.1:11434") # 虚拟机的需要改ip
response = client.chat(model="qwen2.5:7b", messages=[{"role": "user", "content": "你是什么模型？"}])
print(response["message"]['content'])

我是Qwen，一个由阿里云开发的语言模型。我基于大规模预训练技术，能够用于回答问题、创作文字，还能完成多种语言任务。如果您有任何问题或需要帮助，欢迎随时向我提问！


In [3]:
## 流式调用
stream = client.chat(model="qwen2.5:7b", messages=[{"role": "user", "content": "你叫什么名字？"}], stream=True)
for chunk in stream:
    print(chunk["message"]['content'], end="")

我是Qwen，由阿里云开发的人工智能模型。如果您有任何问题或需要帮助，请随时告诉我！

## langchain调用

In [4]:
# from langchain.llms import Ollama
from langchain_community.llms import Ollama

llm = Ollama(base_url="http://127.0.0.1:11434", model="qwen2.5:7b")
llm.invoke("你叫什么名字？")


/Users/chan/projects/Itcast_qa_system/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/yx/_95bz00179jcztsdlyrf1jq00000gn/T/ipykernel_97818/911421712.py:4: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(base_url="http://127.0.0.1:11434", model="qwen2.5:7b")


'我是Qwen，由阿里云开发的人工智能模型。如果您有任何问题或需要帮助，我会尽力提供支持！'

## HTTP调用

In [6]:
# 发送POST
# POST GET
import requests

url = "http://127.0.0.1:11434/api/chat"
headers = {"Content-Type": "application/json"} # 请求头
data = {"model": "qwen2.5:7b", "messages": [{"role": "user", "content": "你叫什么名字？"}], "stream": False}
response = requests.post(url, headers=headers, json=data)
print(response.json())


{'model': 'qwen2.5:7b', 'created_at': '2026-06-15T09:45:41.551092Z', 'message': {'role': 'assistant', 'content': '我是Qwen，由阿里云开发的人工智能模型。如果您有任何问题或需要帮助，请随时告诉我！'}, 'done': True, 'done_reason': 'stop', 'total_duration': 4215098333, 'load_duration': 2197200625, 'prompt_eval_count': 34, 'prompt_eval_duration': 387258000, 'eval_count': 24, 'eval_duration': 1628297000}


In [7]:
url = "http://127.0.0.1:11434/api/chat"
headers = {"Content-Type": "application/json"}
data = {"model": "qwen2.5:7b", "messages": [{"role": "user", "content": "你叫什么名字？"}], "stream": True}
response = requests.post(url, headers=headers, json=data)

for chunk in response.iter_lines():
    print(chunk.decode("utf-8"))

{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.395126Z","message":{"role":"assistant","content":"我是"},"done":false}
{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.465396Z","message":{"role":"assistant","content":"Q"},"done":false}
{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.528909Z","message":{"role":"assistant","content":"wen"},"done":false}
{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.585871Z","message":{"role":"assistant","content":"，"},"done":false}
{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.643847Z","message":{"role":"assistant","content":"由"},"done":false}
{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.702361Z","message":{"role":"assistant","content":"阿里"},"done":false}
{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.758989Z","message":{"role":"assistant","content":"云"},"done":false}
{"model":"qwen2.5:7b","created_at":"2026-06-15T09:45:53.821646Z","message":{"role":"assistant","content":"开发"},"done":false}
{"m

## openai sdk调用Ollama

In [8]:
# !pip install openai SDK HTTP
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="xx") # vllm部署可以指定api_key

stream = client.chat.completions.create(
    model="qwen2.5:7b",
    messages=[{"role": "user", "content": "坦克有后视镜吗"}],
    stream=True,
)
# vllm
for chunk in stream:
    if chunk.choices and chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

print()  # 换行


现代坦克通常并不配备传统的后视镜。这是因为坦克在战场上或复杂地形中行驶时，传统的后视镜可能无法提供足够的视野范围和清晰度，特别是在烟雾、尘土或其他不利条件下。

不过，现代化的坦克装备了先进的观察与导航系统以弥补这一不足，例如：

1. **潜望镜**：包括激光制导的光学装置，使坦克驾驶员可以从车体内部看到外界情况，并且可以通过电子手段增强显示效果。
2. **红外和热成像摄像头**：这些技术可以在低能见度条件下（如夜间或恶劣天气）提供视觉反馈。它们能够检测敌人或障碍物，从而帮助司机安全地驾驶。
3. **昼夜观察仪**：这是一套整合了多种传感器的技术系统，不仅可以提供清晰的白天图像，也可以用于夜视。

综上所述，虽然传统后视镜在坦克中并不常见，但现代坦克装备了许多先进的技术来确保驾驶员的视野和安全性。
